In [ ]:
import pandas as pd
import glob
import os
from tqdm import tqdm

path_1 = r"C:\Percorso\Dati\Anagrafica_Master.xlsx"
path_2 = r"C:\Percorso\Dati\Log_Operativi\*.xlsx"
dir_output_base = r"C:\Percorso\Output\Report_Finali"

df_1 = pd.read_excel(path_1, usecols=['ID_CODICE', 'CATEGORIA', 'AREA', 'PROPRIETARIO', 'TIPOLOGIA_ASSET', 'INDIRIZZO', 'CLASSE_UTENZA'])
df_1['ID_CODICE'] = df_1['ID_CODICE'].astype(str).str.strip()

esclusioni = [
    'MODELLO_X_LARGE', 
    'MODELLO_Y_LARGE', 
    'VERSIONE_INDUSTRIALE',
    'CATEGORIA_SPECIALE_V1',
    'CATEGORIA_SPECIALE_V2'
]

df_1 = df_1[~df_1['TIPOLOGIA_ASSET'].isin(esclusioni)]

lista_file_input = glob.glob(path_2)
df_2 = pd.concat([pd.read_excel(f) for f in tqdm(lista_file_input)], ignore_index=True)

df_2['DATA_RILEVAZIONE'] = pd.to_datetime(df_2['DATA_RILEVAZIONE'], dayfirst=True, errors='coerce')
df_2 = df_2.dropna(subset=['DATA_RILEVAZIONE'])

mesi_nomi = {
    1: "Gennaio", 2: "Febbraio", 3: "Marzo", 4: "Aprile", 5: "Maggio", 6: "Giugno",
    7: "Luglio", 8: "Agosto", 9: "Settembre", 10: "Ottobre", 11: "Novembre", 12: "Dicembre"
}

data_ref = df_2['DATA_RILEVAZIONE'].iloc[0]
mese_str = mesi_nomi[data_ref.month]
anno_str = data_ref.year

df_2['ID_RILEVATO'] = df_2['ID_RILEVATO'].astype(str).str.strip()

df_1_2 = pd.merge(
    df_1, 
    df_2[['ID_RILEVATO']], 
    left_on='ID_CODICE', 
    right_on='ID_RILEVATO', 
    how='left'
)

df_3 = df_1_2[df_1_2['ID_RILEVATO'].isna()].copy()
df_3 = df_3.drop(columns=['ID_RILEVATO']).drop_duplicates(subset=['ID_CODICE'])

folder_res = os.path.join(dir_output_base, "Analisi_Assenze_Mensili")
os.makedirs(folder_res, exist_ok=True)

file_res = os.path.join(folder_res, f"Report_Assenze_{mese_str}_{anno_str}.xlsx")

with pd.ExcelWriter(file_res, engine='xlsxwriter') as writer:
    df_3.to_excel(writer, index=False, sheet_name='Elementi_Non_Rilevati')
    
    workbook  = writer.book
    worksheet = writer.sheets['Elementi_Non_Rilevati']
    
    (max_row, max_col) = df_3.shape
    col_names = [{'header': col} for col in df_3.columns]
    
    worksheet.add_table(0, 0, max_row, max_col - 1, {
        'columns': col_names,
        'style': 'Table Style Medium 9',
        'name': 'TabellaRilevazioni'
    })

    for i, col in enumerate(df_3.columns):
        width = max(df_3[col].astype(str).map(len).max(), len(col)) + 2
        worksheet.set_column(i, i, min(width, 50))

dir_storico = r"C:\Percorso\Output\Report_Finali\Archivio_Storico"
path_storico = os.path.join(dir_storico, "*.xlsx")
file_output_critici = os.path.join(dir_storico, "ANOMALIE_CRITICHE_RICORRENTI.xlsx")

lista_storico = [f for f in glob.glob(path_storico) if "ANOMALIE_CRITICHE" not in f]

if len(lista_storico) >= 2:
    set_comuni = None
    df_anagrafica_completa = None

    for f in lista_storico:
        df_temp = pd.read_excel(f)
        if 'ID_CODICE' in df_temp.columns:
            df_temp['ID_CODICE'] = df_temp['ID_CODICE'].astype(str).str.strip()
            ids_attuali = set(df_temp['ID_CODICE'])

            if set_comuni is None:
                set_comuni = ids_attuali
                df_anagrafica_completa = df_temp.drop_duplicates(subset=['ID_CODICE'])
            else:
                set_comuni = set_comuni.intersection(ids_attuali)

    if set_comuni:
        df_4 = df_anagrafica_completa[df_anagrafica_completa['ID_CODICE'].isin(set_comuni)].copy()
        
        with pd.ExcelWriter(file_output_critici, engine='xlsxwriter') as writer:
            df_4.to_excel(writer, index=False, sheet_name='Criticità_Persistenti')
            
            workbook  = writer.book
            worksheet = writer.sheets['Criticità_Persistenti']
            (r, c) = df_4.shape
            
            settings = [{'header': col} for col in df_4.columns]
            worksheet.add_table(0, 0, r, c - 1, {
                'columns': settings,
                'style': 'Table Style Medium 3', 
                'name': 'TabellaCritici'
            })
            
            for i, col in enumerate(df_4.columns):
                width = max(df_4[col].astype(str).map(len).max(), len(col)) + 2
                worksheet.set_column(i, i, min(width, 50))

print("Processo terminato con successo.")

In [ ]:
Riassunto del Codice (Ideale per il tuo Portfolio)
Questo script Python è una soluzione di Data Auditing automatizzata progettata per il controllo qualità e il monitoraggio degli asset. Il processo si articola in tre fasi principali:

Data Cleaning & Filtering: Il software carica un database master (es. inventario asset) e una serie di log operativi grezzi (PWT). Applica filtri avanzati per escludere categorie non rilevanti e normalizza i codici identificativi per garantire l'integrità del confronto.

Gap Analysis Mensile: Attraverso un'operazione di left join tra il database master e i log di attività, il codice identifica automaticamente tutti gli asset che non hanno registrato attività nel periodo di riferimento. I risultati vengono esportati in un file Excel formattato professionalmente con tabelle dinamiche e auto-adattamento delle colonne.

Analisi Storica delle Ricorrenze (Cross-Check): La funzione più avanzata dello script esegue un confronto tra i report dei mesi precedenti. Utilizzando la logica degli insiemi (intersezione), individua gli "Asset Critici", ovvero quegli elementi che risultano assenti in modo sistematico in tutti i periodi analizzati. Questo permette di distinguere tra anomalie temporanee e problemi infrastrutturali persistenti.

Tecnologie utilizzate: Pandas per la manipolazione dati, XlsxWriter per il reporting avanzato, Glob e OS per la gestione dinamica del file system.